INTRO PARAGRAPH GOES HERE

Plan:
* What is Spark?
  * Spark has many different feautres, but RDDs are fundamental
  * RDDs versus standard processing
  * Different APIs
* Dataframes
  * Way to structure data similar to a spreadsheet (columns, rows)
  * Introduced in API version ...
  * RDDs under the hood (Spark manages complexity)
* Using Spark Dataframes
  * Basic example

## Spark

Spark is an open-source framework used for processing large amounts of data quickly and reliably. 

The Spark engine is written in Scala. To interface with the Spark engine there are several supported APIs in different languages, including Java, Scale, Python and R. For this tutorial we'll be using the Python Spark API, known as PySpark.



## Processing Data

Before we go into the details of using Spark's Python API, let's take a step back and look at some of the issues that it was designed to solve.

When processing data a number of transformations are often applied in order to get it into a form that is useful. This can include transformations like filtering, grouping and mapping.

For example take a dashboard that displays sales information for the past week. Each sale is stored as a single record in a database table. It will include the date, cost, shipping and customer for each record. By transforming the data, records can be grouped by date, with the sum of sales calculated for each date. 

In a system where only one computer is used processing this data might look like this:

![Data Flow](images/spark-intro/etl_1.svg)


Data is fed in at one end (in this case sales data), a bunch of transformations are applied to the data in the middle, and then the data is output at the other end (in our example it is sent to a reporting dashboard). This system is often referred to as Extract, Transform, Load (ETL).

There are a few limitations to this approach of using a single machine to process large datasets:
1. Data is often processed synchronously. For large sets of data this can be slow.
1. When data is processed in parallel or asynchronously, this can lead to complex code or unforseen bugs like race conditions.
1. The system is not very fault tolerant. If the single computer crashes, the progress made transforming the data is lost. 

To overcome these issues, Spark uses a techique called a Resilient Distributed Dataset (RDD).

### Resilient Distributed Datasets

As with a standard ETL system, Spark takes a data input, transforms the data, and then loads it to another destination. The most significant difference is that Spark does transformation across a cluster of computers, instead of just one.

To process the data across a cluster of computers, Spark breaks the datasets into chunks. This allows the data to be processed in parallel on several machines, after which it is recombined into a single dataset. This technique is called a Resilient Distributed Dataset (RDD) and is fundamental to how Spark works.

Here is a modified version of the ETL diagram that shows the data being split into chunks and distributed:

![Data Flow](images/spark-intro/rdd_1.svg)

By chunking and distributing data in this way, Spark is able to process large datasets very quickly. It also means that it can scale up as the data gets larger by adding more computers to the cluster.

In addition to chunking and distributing data, Spark also does one other thing to the data in an RDD: it duplicates it. Before distributing the data across the cluster, Spark duplicates each chunk. The reason it does this is to protect against computers in the cluster failing. If a computer crashes while it is processing a chunk of data, it doesn't matter, the same chunk of data can be sent to be processed by another healthy computer in the cluster. 

By duplicating data with RDDs, the system is much more resilient to failures.


### Dataframes

Now that we know the concepts behind processing data with Spark's RDDs, lets look at how to use them.

Spark uses Dataframes to represent data. Dataframes are structured like a spreadsheet: they have a rows for data items and columns for each attribute of the data. Transformations can be applied to data directly from the dataframe using a verity of methods including filtering, grouping, agrregation and mapping.

Dataframes use RDD features under the hood. This means that the transformations applied to dataframes can be done across a cluster of computers, taking advantage of the speed and reslience of RDDs. Spark manages all of this complexity, allowing the programmer to define how they want the data to be processed, without worrying about distributing the data across a cluster of computers.

As well as running Spark on a cluster, we're able to run Spark on a single computer without any differences to the code. This means that you can develop your Spark code locally before you run it on a cluster in production.

Let's take a look using Dataframes in PySpark.

## PySpark

For this example I'll be using some fake sales data stored in a csv. I've made [the data available to download](/static/sales_data.csv) if you want to follow along with the example.


To get started with Spark's Python API you need to install PySpark:

``` bash
pip install pyspark
```

Once `pyspark` is successfully installed we can start using Spark.

First we need to import pyspark and set up a spark session:


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('pokemon').getOrCreate()

This will start a new Spark app, or get an existing one if it already exists. I've named my Spark app `sales` in this case.

Now that I've started the session I can create a dataframe with my sales data:

In [2]:
pokemon = spark.read.json('static/spark-intro/pokemon.json')

Spark comes with several methods to read common file types. Here I've used the `csv()` read method. There are also methods for json and text files, among many others.

The `inferSchema=True` argument means that Spark will automatically infer the data types for each column when the data is loaded. The `header=True` argument tells spark that the first row of the data contains the headers for each column.

Now that I've loaded the data into a dataframe, lets see what it looks like using the `show()` method:

In [3]:
pokemon.show()

+------+-------+------+---+----------+-----+--------------+---------------+-----+------+------+------+
|attack|defense|height| hp|      name|order|special-attack|special-defense|speed|type_1|type_2|weight|
+------+-------+------+---+----------+-----+--------------+---------------+-----+------+------+------+
|    49|     49|     7| 45| bulbasaur|    1|            65|             65|   45| grass|poison|    69|
|    62|     63|    10| 60|   ivysaur|    2|            80|             80|   60| grass|poison|   130|
|    82|     83|    20| 80|  venusaur|    3|           100|            100|   80| grass|poison|  1000|
|    52|     43|     6| 39|charmander|    5|            60|             50|   65|  fire|  null|    85|
|    64|     58|    11| 58|charmeleon|    6|            80|             65|   80|  fire|  null|   190|
|    84|     78|    17| 78| charizard|    7|           109|             85|  100|  fire|flying|   905|
|    48|     65|     5| 44|  squirtle|   10|            50|             6

In [4]:
pokemon.count()

949

We can also view the schema for the table to check the data types that Spark has inferred for each column, using the `printSchema()` method:

In [5]:
pokemon.printSchema()

root
 |-- attack: long (nullable = true)
 |-- defense: long (nullable = true)
 |-- height: long (nullable = true)
 |-- hp: long (nullable = true)
 |-- name: string (nullable = true)
 |-- order: long (nullable = true)
 |-- special-attack: long (nullable = true)
 |-- special-defense: long (nullable = true)
 |-- speed: long (nullable = true)
 |-- type_1: string (nullable = true)
 |-- type_2: string (nullable = true)
 |-- weight: long (nullable = true)



Using the `.select()` method I can define which columns I want:

In [6]:
pokemon.select('name', 'type_1').show()

+----------+------+
|      name|type_1|
+----------+------+
| bulbasaur| grass|
|   ivysaur| grass|
|  venusaur| grass|
|charmander|  fire|
|charmeleon|  fire|
| charizard|  fire|
|  squirtle| water|
| wartortle| water|
| blastoise| water|
|  caterpie|   bug|
|   metapod|   bug|
|butterfree|   bug|
|    weedle|   bug|
|    kakuna|   bug|
|  beedrill|   bug|
|    pidgey|normal|
| pidgeotto|normal|
|   pidgeot|normal|
|   rattata|normal|
|  raticate|normal|
+----------+------+
only showing top 20 rows



Let's run through some common transformations on the sales data.



### Filtering

Filtering can be used to quickly return a subset of the data set. Here I am using the `filter()` method to retrieve all of the orders in the United States:

In [7]:
pokemon.filter(pokemon.type_1 == 'dragon') \
    .select('name', 'type_1') \
    .show()

+---------+------+
|     name|type_1|
+---------+------+
|  dratini|dragon|
|dragonair|dragon|
|dragonite|dragon|
|  altaria|dragon|
|    bagon|dragon|
|  shelgon|dragon|
|salamence|dragon|
|   latias|dragon|
|   latios|dragon|
| rayquaza|dragon|
|    gible|dragon|
|   gabite|dragon|
| garchomp|dragon|
|     axew|dragon|
|  fraxure|dragon|
|  haxorus|dragon|
|druddigon|dragon|
| reshiram|dragon|
|   zekrom|dragon|
|   kyurem|dragon|
+---------+------+
only showing top 20 rows



In [8]:
pokemon \
    .filter((pokemon.type_1 == 'dragon') | (pokemon.type_2 == 'dragon')) \
    .select('name', 'type_1', 'type_2') \
    .show()

+----------------+------+-------+
|            name|type_1| type_2|
+----------------+------+-------+
|         dratini|dragon|   null|
|       dragonair|dragon|   null|
|       dragonite|dragon| flying|
|         kingdra| water| dragon|
|         vibrava|ground| dragon|
|          flygon|ground| dragon|
|         altaria|dragon| flying|
|           bagon|dragon|   null|
|         shelgon|dragon|   null|
|       salamence|dragon| flying|
|          latias|dragon|psychic|
|          latios|dragon|psychic|
|        rayquaza|dragon| flying|
|           gible|dragon| ground|
|          gabite|dragon| ground|
|        garchomp|dragon| ground|
|          dialga| steel| dragon|
|          palkia| water| dragon|
|giratina-altered| ghost| dragon|
|            axew|dragon|   null|
+----------------+------+-------+
only showing top 20 rows



Spark uses a special syntax in this case to define which column I want to use when filtering `sales_data['location']`. It is similar to accessing a value of a dictionary. Unlike a dictionary which returns a single value, it returns a column that Spark will use when checking which rows match the filter condition. In this case the condition checks the location is equal to `'United States'`.

Each transformation method of a `Dataframe` returns a new `Dataframe`. I can store the result of a transformation in a variable if I want to reuse the result. Here I want to work out the number of large orders (more than 10 items):

In [9]:
tall_pokemon = pokemon.filter(pokemon.height > 20)

Since I stored the result of the filter transformation in a variable I can reuse it. First let's see the number of orders:

In [10]:
tall_pokemon.count()

90

Now let's look at first item in the data:

In [11]:
tallest_pokemon = tall_pokemon \
    .sort(pokemon.height.desc()) \
    .first()
    
tallest_pokemon

Row(attack=90, defense=45, height=145, hp=170, name='wailord', order=406, special-attack=90, special-defense=45, speed=60, type_1='water', type_2=None, weight=3980)

Here I've used the `.first()` method to retrieve the first row in the dataset. This returns a `Row` object, which behaves like a Python object and a Python dictionary:

In [12]:
tallest_pokemon['name']

'wailord'

In [13]:
tallest_pokemon.name

'wailord'

### Aggregations and Calculations

Quite often when processing data you will want to calculate a value from existing columns. You probably also want to aggregate several rows together into a single value, like a sum or an average.

Spark provides these featues in variety of different ways. 

First let's look at using the `select()` method. In this case I want to see which countries customers have placed orders from:

In [14]:
pokemon.select('type_1') \
    .distinct() \
    .show()

+--------+
|  type_1|
+--------+
|     bug|
|  normal|
|   ghost|
|   grass|
|   steel|
|     ice|
|   water|
|  ground|
|  flying|
|   fairy|
|    dark|
|fighting|
|  dragon|
|  poison|
| psychic|
|    rock|
|electric|
|    fire|
+--------+



To start I tell Spark which columns I want to include using the `select()` method. Then I use the `distinct()` method to remove duplicate values from the result. 

Next I want to see the total sales for the data:

In [15]:
from pyspark.sql.functions import sum

pokemon.select(sum('height')) \
    .show()

+-----------+
|sum(height)|
+-----------+
|      11656|
+-----------+



I give the column name that I want to total 0f to the `sum()` method. This is called from inside of the `select()` method to return only the calculated value.

I can also see the average sales for the data using the `avg()` method:

In [16]:
from pyspark.sql.functions import avg

pokemon.select(avg('hp')) \
    .show()

+-----------------+
|          avg(hp)|
+-----------------+
|68.95468914646997|
+-----------------+



In [17]:
from pyspark.sql.functions import max

pokemon.select(max('hp')) \
    .show()

+-------+
|max(hp)|
+-------+
|    255|
+-------+



### Grouping

Finally, let's look at grouping data. Using the `groupBy()` method I can consolidate data into related groups. After I have grouped the data I can perform aggregations and other transormations on the data.

In this example I want to see the total sales. I use the `groupBy()` method to group the data into countries:

In [18]:
from pyspark.sql.functions import round, desc

pokemon_types = pokemon.groupBy(pokemon['type_1'])

pokemon_types.max() \
    .select('type_1', 'max(hp)') \
    .sort(desc('max(hp)')) \
    .show()

+--------+-------+
|  type_1|max(hp)|
+--------+-------+
|  normal|    255|
|    dark|    223|
|  dragon|    216|
| psychic|    190|
|   water|    170|
|   ghost|    150|
|fighting|    144|
|   fairy|    126|
|   grass|    123|
|    rock|    123|
|  ground|    115|
|    fire|    115|
|     ice|    110|
|     bug|    107|
|  poison|    105|
|   steel|    100|
|electric|     90|
|  flying|     85|
+--------+-------+



After I have grouped the data by country, I calculate the sum of all the columns using the `sum()` method. I then select only the columns I want to display. The `format_number()` method is used to make sure the total sales are only displayed to two decimal places.

Applying transformations to columns gives them long and cumbersome names. I use the `withColumnRenamed()` method to make rename the calculated total cost column to `total sales`. If I didn't do this it would be called `format_number(sum(total_cost), 2)` in the output.

In this final example I want to group sales by the day of the week:

In [19]:
pokemon_types.avg() \
    .select('type_1', round('avg(attack)', 2).alias('avg attack')) \
    .sort(desc('avg attack')) \
    .show()

+--------+----------+
|  type_1|avg attack|
+--------+----------+
|  dragon|    108.67|
|fighting|      99.1|
|  ground|     96.22|
|   steel|     93.13|
|    rock|     89.98|
|    dark|      84.7|
|    fire|     84.25|
|  flying|     78.75|
|   ghost|      76.3|
|  normal|     75.86|
|   grass|     75.15|
|   water|     74.71|
|  poison|     73.54|
|     ice|     73.07|
| psychic|     72.55|
|     bug|     72.49|
|electric|     68.15|
|   fairy|     61.21|
+--------+----------+



The only new thing in this example is the use of the `date_format` function. I use it to transform the date into a human readable day of the week.

# Summary
